In [12]:
import numpy as np
import matplotlib.pyplot as plt
import napari
from skimage.filters import threshold_otsu, threshold_minimum, try_all_threshold, threshold_mean, gaussian
import tifffile
from skimage.exposure import rescale_intensity
from skimage.filters import threshold_multiotsu
from skimage.measure import label, regionprops
from skimage.measure import regionprops_table
import pandas as pd
from skimage.morphology import remove_small_objects
%matplotlib inline


In [2]:
image = tifffile.imread(r"C:\Users\isobe\The University of Manchester Dropbox\Isobel Taylor-Hearn\curationcopy\out\050726_Wide_Reservoir_+_flow_day_10_R_2_Merged\050726_Wide_Reservoir_+_flow_day_10_R_2_Merged_6channel.tif")

In [26]:
bf = image[:, 0, :, :].astype(float)

fluo = gaussian(rescale_intensity(
    image[:, 2, :, :],
    out_range=(0, 255)
).astype(np.uint8), sigma=1)

mask = image[:, -2, :, :].astype(bool)

# Only use fluorescence values inside the mask to calculate thresholds
masked_values = fluo[mask]

thresholds = threshold_multiotsu(masked_values, classes=3)

# Middle + high fluorescence classes = fluorescence positive
fluorescent_positive = (remove_small_objects(label(mask & (fluo > thresholds[0])), min_size=1e5))>0
# fluorescent_positive = remove_small_objects(label(mask & (fluo > threshold_otsu(masked_values))), min_size=1e6)

In [28]:
viewer = napari.Viewer()
viewer.add_image(fluo, name='Fluo')
viewer.add_labels(mask.astype(np.uint8), name='Mask')
viewer.add_labels(fluorescent_positive, name='Perfused')
viewer.add_image(bf, name='BF')

<Image layer 'BF' at 0x20818557e80>